# Market Data Visualization

Interactive visualization of real-time and historical market data using project integration modules.


In [ ]:
# Install/check dependencies
required_packages = {
    'pandas': 'pandas>=2.1.0',
    'numpy': 'numpy>=1.24.0',
    'plotly': 'plotly>=5.18.0',
}

missing_packages = []
for package, requirement in required_packages.items():
    try:
        __import__(package)
        print(f"✓ {package} is installed")
    except ImportError:
        missing_packages.append(requirement)
        print(f"✗ {package} is missing")

if missing_packages:
    print(f"\n⚠ Missing packages: {', '.join(missing_packages)}")
    print("Install with: pip install -r requirements-notebooks.txt")
else:
    print("\n✓ All required packages are installed")


## Setup and Imports


In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta
import os

# Add project to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "python"))

print(f"Project root: {project_root}")

# Import data analysis libraries
try:
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print("✓ Successfully imported data analysis libraries")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("Please run the dependency check cell above and install missing packages:")
    print("  pip install -r requirements-notebooks.txt")
    raise


In [ ]:
# Import project modules
try:
    from integration.config_loader import ConfigLoader
    from integration.alpaca_client import AlpacaClient
    from integration.ibkr_portal_client import IBKRPortalClient
    print("✓ Successfully imported project modules")
except ImportError as e:
    print(f"⚠ Import error: {e}")
    print("Note: Some features may not be available")


## Connect to Market Data Sources


In [ ]:
# Initialize Alpaca client (if credentials available)
alpaca_client = None
try:
    if os.getenv("ALPACA_API_KEY_ID") and os.getenv("ALPACA_API_SECRET_KEY"):
        alpaca_client = AlpacaClient()
        print("✓ Alpaca client initialized")
    else:
        print("⚠ Alpaca credentials not found in environment")
        print("  Set ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY to use Alpaca data")
except Exception as e:
    print(f"⚠ Failed to initialize Alpaca client: {e}")

# Initialize IBKR Portal client (if available)
ibkr_client = None
try:
    from integration.ibkr_portal_client import IBKRPortalClient
    ibkr_client = IBKRPortalClient()
    print("✓ IBKR Portal client initialized")
except Exception as e:
    print(f"⚠ IBKR Portal client not available: {e}")


## Real-Time Price Chart (Candlestick)


In [ ]:
# Example: Get snapshot data and create candlestick chart
symbol = "SPY"

if alpaca_client:
    try:
        snapshot = alpaca_client.get_snapshot(symbol)
        print(f"Snapshot for {symbol}:")
        print(f"  Last: ${snapshot.get('last', 'N/A')}")
        print(f"  Bid: ${snapshot.get('bid', 'N/A')}")
        print(f"  Ask: ${snapshot.get('ask', 'N/A')}")
        print(f"  Spread: ${snapshot.get('spread', 'N/A')}")
    except Exception as e:
        print(f"Error fetching snapshot: {e}")
        snapshot = None
else:
    print("Alpaca client not available - using sample data")
    snapshot = None

# Create sample candlestick chart (replace with real historical data)
dates = pd.date_range(end=datetime.now(), periods=30, freq='D')
sample_data = pd.DataFrame({
    'date': dates,
    'open': 450 + np.random.randn(30).cumsum(),
    'high': 450 + np.random.randn(30).cumsum() + 2,
    'low': 450 + np.random.randn(30).cumsum() - 2,
    'close': 450 + np.random.randn(30).cumsum() + 1,
    'volume': np.random.randint(1000000, 5000000, 30)
})

fig = go.Figure(data=go.Candlestick(
    x=sample_data['date'],
    open=sample_data['open'],
    high=sample_data['high'],
    low=sample_data['low'],
    close=sample_data['close']
))

fig.update_layout(
    title=f"{symbol} - Candlestick Chart (Sample Data)",
    xaxis_title="Date",
    yaxis_title="Price ($)",
    height=500
)

fig.show()


## Volatility Surface Visualization

Visualize implied volatility across strikes and expiration dates.


In [ ]:
# Example volatility surface (replace with actual option chain data)
strikes = np.arange(400, 500, 5)
expirations = [30, 45, 60, 90, 120]  # Days to expiry

# Generate sample IV data
iv_data = np.random.rand(len(expirations), len(strikes)) * 0.3 + 0.15

fig = go.Figure(data=go.Heatmap(
    z=iv_data,
    x=strikes,
    y=expirations,
    colorscale='Viridis',
    colorbar=dict(title="Implied Volatility")
))

fig.update_layout(
    title="Implied Volatility Surface (Sample Data)",
    xaxis_title="Strike Price ($)",
    yaxis_title="Days to Expiry",
    height=400
)

fig.show()


## Next Steps

- Connect to live option chain data
- Query QuestDB for historical quotes
- Add Greeks visualization (delta, gamma, theta, vega)
- Implement real-time streaming updates
